In [2]:
import os
import pandas as pd
import statsmodels.api as sm
from statsmodels.sandbox.regression.gmm import IV2SLS
import numpy as np

os.chdir('/Users/fogellmcmuffin/Documents/thesis/_replication/')    # Working dir

pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', None)

vintage_trim = pd.read_csv('cleaned_data/vintage_trim.csv')

mean_spf_trim = pd.read_csv('cleaned_data/mean_spf_trim.csv')
ind_spf_trim = pd.read_csv('cleaned_data/ind_spf_trim.csv')
oilp = pd.read_csv('cleaned_data/oil_prices.csv')

mean_spf_trim['DATE'] = pd.to_datetime(mean_spf_trim['DATE'])
ind_spf_trim['DATE'] = pd.to_datetime(ind_spf_trim['DATE'])

ind_spf_trim = ind_spf_trim.dropna(subset='r_t1')

In [3]:
###############################################
                ## Controls ##
###############################################

def inf_control(df, v, end_date):
    v['lt_3'] = v['t3'].shift(1)
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    v = v.loc[(v['DATE'] >= '1981-12-31') & (v['DATE'] <= end_date)]
    L_pi = v['lt_3']
    revisions = df[f'r_t3']
    x = np.column_stack((L_pi, revisions))
    x = sm.add_constant(x)
    errors = df[f'e_t3']
    initial = sm.OLS(errors, x).fit()
    regs = initial.get_robustcov_results(cov_type='HAC', maxlags=None)
    return regs


def oil_control(df, oil, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    oil = oil.loc[(oil['DATE'] >= '1981-12-31') & (oil['DATE'] <= end_date)]
    revisions = df[f'r_t3']
    L_oil = oil['pc_1q']
    x = np.column_stack((L_oil, revisions))
    x = sm.add_constant(x)
    errors = df[f'e_t3']
    initial = sm.OLS(errors, x).fit()
    regs = initial.get_robustcov_results(cov_type='HAC', maxlags=None)
    return regs


date = '2020-06-30'
regs = [inf_control(mean_spf_trim, vintage_trim, date), oil_control(mean_spf_trim, oilp, date)]

regs[1].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   e_t3   R-squared:                       0.080
Model:                            OLS   Adj. R-squared:                  0.068
Method:                 Least Squares   F-statistic:                     8.496
Date:                Thu, 09 May 2024   Prob (F-statistic):           0.000318
Time:                        08:33:16   Log-Likelihood:                 492.00
No. Observations:                 155   AIC:                            -978.0
Df Residuals:                     152   BIC:                            -968.9
Df Model:                           2                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0028      0.001     -2.304      0.023      -0.005      -0.000
x1             0.0229      0.007      3.485      0.001       0.010       0.036
x2            -0.2338      0.326     -0.717      0.474      -0.878       0.410
==============================================================================
Omnibus:                       11.399   Durbin-Watson:                   0.633
Prob(Omnibus):                  0.003   Jarque-Bera (JB):               16.504
Skew:                          -0.414   Prob(JB):                     0.000261
Kurtosis:                       4.368   Cond. No.                         266.
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using None lags and without small sample correction
"""

In [9]:
###############################################
         ## Heterogeneity in Priors ##
###############################################

### Individual Forecasts ###
def E3R2_PLD(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    revisions = df['r_t2']
    revisions = sm.add_constant(revisions)
    errors = df[f'e_t3']
    regs = sm.OLS(errors, revisions).fit(cov_type='cluster', cov_kwds = {"groups":df['ID']})
    return regs


### Mean Forecasts ###
def IV_OIL_INF(df, oil, v, end_date):
    v['lt_3']= ((v['t']/400 + 1)*(v['t1']/400 + 1)*(v['t2']/400 + 1)*(v['t3']/400 + 1)-1)
    L_pi = v['lt_3'].shift(1)
    df = df.loc[(df['DATE'] >= '1986-09-30') & (df['DATE'] <= end_date)]
    oil = oil.loc[(oil['DATE'] >= '1986-09-30') & (oil['DATE'] <= end_date)]
    v = v.loc[(v['DATE'] >= '1986-09-30') & (v['DATE'] <= end_date)]
    L_pi = v['lt_3']
    revisions = df[f'r_t3']
    x = np.column_stack((revisions, L_pi))
    x = sm.add_constant(x)
    errors = df[f'e_t3']
    L1_oil = oil['pc_1q']
    L2_oil = oil['pc_2q']
    w = np.column_stack((L_pi, L1_oil, L2_oil))
    w = sm.add_constant(w)
    initial = IV2SLS(errors, x, instrument = w).fit()
    reg = initial.get_robustcov_results(cov_type='HAC', maxlags=None)
    return reg
    
    
date = '2020-06-30'
regs = [E3R2_PLD(ind_spf_trim, date), IV_OIL_INF(mean_spf_trim, oilp, vintage_trim, date)]

regs[1].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                          IV2SLS Regression Results                           
==============================================================================
Dep. Variable:                   e_t3   R-squared:                      -0.075
Model:                         IV2SLS   Adj. R-squared:                 -0.091
Method:                     Two Stage   F-statistic:                     1.081
                        Least Squares   Prob (F-statistic):              0.342
Date:                Thu, 09 May 2024                                         
Time:                        09:09:26                                         
No. Observations:                 136                                         
Df Residuals:                     133                                         
Df Model:                           2                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0017      0.004      0.410      0.682      -0.006       0.010
x1             0.6658      0.626      1.064      0.289      -0.572       1.903
x2            -0.0049      0.009     -0.539      0.591      -0.023       0.013
==============================================================================
Omnibus:                       16.259   Durbin-Watson:                   0.831
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               40.299
Skew:                          -0.401   Prob(JB):                     1.77e-09
Kurtosis:                       5.544   Cond. No.                         295.
==============================================================================
"""